# ZMART target acquisition — React edition

The same run as `zmart_microscopy_v4.ipynb`, with the four review steps as live React apps inside the cells: every tile, focus point, and image pair appears the moment the microscope produces it, and the buttons in the browser drive the kernel (which drives the hardware through the same gated paths as always).

Needs the `anywidget` package in the kernel, and internet access in the **browser** the first time a widget renders (React loads from the esm.sh CDN). If the browser is offline, the widget cell shows a note saying so — working offline, use `zmart_microscopy_v4.ipynb`, the matplotlib edition of the exact same workflow. Run the cells top to bottom.

## 1 · Setup and connect
Edit the analysis repo, then run.

In [ ]:
import sys
from pathlib import Path as _Path
_target_acq = _Path("workflows/target_acquisition")
if not _target_acq.is_dir():
    _target_acq = _Path.cwd()
sys.path.insert(0, str(_target_acq.resolve()))
from _bootstrap import Path, workflow
from workflow import react as wreact

ANALYSIS_REPO = Path("C:/code/smart-analysis")
SIMULATE_IMAGES = False
AUTOFOCUS_JOB = None  # Set only when LAS X has more than one autofocus job.

if "engine" in globals():
    try:
        engine.shutdown()
    except Exception as exc:
        print(f"warning: the previous analysis engine did not shut down cleanly: {exc}")
    del engine
if "zmart_controller" in globals():
    try:
        zmart_controller.disconnect()
    except Exception as exc:
        print(f"warning: the previous session did not disconnect cleanly: {exc}")
    del zmart_controller

engine = workflow.load_analysis_engine(ANALYSIS_REPO)
try:
    workflow.preflight_analysis_engine(engine)
    zmart_controller = workflow.connect("leica")
    ROOT = Path(zmart_controller.run_procedure({"name": "get_root"})["root"])
except Exception:
    try:
        if "zmart_controller" in globals():
            zmart_controller.disconnect()
            del zmart_controller
    finally:
        engine.shutdown()
        del engine
    raise
ROOT

## 2 · Set origin
Move to the run origin first. This makes the current position `(0, 0, 0)`.

In [ ]:
zmart_controller.set_origin()

## 3a · Overview job

Select the low-magnification overview job in LAS X, then run.

In [ ]:
overview_state = zmart_controller.get_state()
limits = overview_state["observed"].get("limits")
if not limits or limits.get("is_fallback") or limits.get("source") != "machine":
    raise RuntimeError(
        f"machine-specific stage limits are not active (got {limits}); publish this "
        "machine's measured envelope with limits/notebooks/set_stage_limits.ipynb first"
    )
overview_state["observed"]

## 3b · Target job

Select the high-magnification target job in LAS X, then run.

In [ ]:
target_state = zmart_controller.get_state()
if target_state["changeable"].get("job") == overview_state["changeable"].get("job"):
    raise RuntimeError("the overview and target jobs are the same; select the high-magnification target job")
target_state["observed"]

## 4 · Initial positions
Read overview tile centres from the controller.

In [ ]:
positions = zmart_controller.run_procedure({"name": "get_positions"})["positions"]
print(len(positions), "overview positions")

## 5 · Focus

Click the map to add focus points (points already placed in LAS X are pre-filled; click a point to remove it), then press **Measure focus**. The stage autofocuses at each point and the fitted focus map streams in live, refining after every point. The map is drawn with +x to the right and +y downwards, the same orientation as the overview map below.

Re-measuring after editing points only visits the new or moved points — everything already measured this session is reused. If the focus may have drifted (a long pause, a bumped stage), press **Measure fresh** instead: it forgets this session's measurements and re-drives every point. Once measured, the overview tile markers take the colour of the fitted focus map.

In [ ]:
picker = wreact.pick_focus_points(
    zmart_controller, positions, af_job=AUTOFOCUS_JOB
)
picker

In [ ]:
focus = picker.require_focus()
workflow.plot_focus_surface(focus, save_path=ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 5b · Validate the objective calibration

Before trusting the run's cross-objective moves, measure how far the XY calibration is actually off — on this stage, today. The check visits 12 sites on a ring about 1 mm from the origin, images each with the overview job (objective 1), then returns to the *same frame positions* with the target job (objective 2; the driver applies the calibrated translation) and images them again. Registering each pair gives the leftover offset per site; averaging 12 independent stage moves separates the systematic calibration error (the mean) from the stage's own move-to-move scatter.

Run the two cells in order. The report and plot are written into the run folder; `mean_offset_um` is how far the calibration is off, `stage_scatter_rms_um` how repeatable the stage was.

Reading the sign: a positive `mean_dx_um` means objective 2 landed that many micrometres towards +x of where objective 1 imaged the same position (and likewise for y). If the mean is larger than you can accept — say, a cell radius — re-run the objective-pair calibration; the mean is exactly what the objective-2 translation is off by.

Two things can stop the check early, both before or without acquiring anything wrong: a ring site outside this machine's measured stage envelope (re-run with a smaller `radius_um`), and focus points that do not reach the ring (the fitted focus surface would be guessing out there — add focus points that cover the ring, or shrink `radius_um`).

In [ ]:
calcheck = workflow.start_calibration_check(
    zmart_controller, overview_state, focus=focus, n_positions=12, radius_um=1000.0
)
print(len(calcheck.reference_records), "objective-1 reference images acquired")

In [ ]:
calibration_report = workflow.finish_calibration_check(
    calcheck, target_state, output_root=ROOT
)
{k: v for k, v in calibration_report.items() if k != "sites"}

## 6 · Overview

Capture one overview at each position and watch the map grow: every tile appears at its real stage position the moment it is saved, while the microscope scans the rest. Drag the map to pan and scroll to zoom (the view stops auto-fitting once you do — press **Fit** to frame everything again); the readout under the map shows the cursor's frame position in micrometres. The controls on the right work per channel: the colour swatch cycles the palette, **on/off** toggles visibility, and the min/max boxes set the display range for brightness and contrast (the same idea as Fiji's B&C — type a value and press Enter or click away to apply).

In [ ]:
viewer = wreact.view_overview()
display(viewer)
overview_records = workflow.run_overview(
    zmart_controller, positions, state=overview_state, focus=focus,
    on_record=viewer.add_acquisition,
)
print(len(overview_records), "overviews captured")
if SIMULATE_IMAGES:
    n = workflow.hijack_if_simulating(overview_records, simulate=True)
    viewer.reload()  # the hijack rewrote the saved pixels
    print(n, "overview images replaced")

In [ ]:
overviews = workflow.overview_inputs_from_records(positions, overview_records, focus=focus)

## 7 · Discover targets

Segment the overview images, then decide which cells become targets in the explorer:

- the two dropdowns put any measured feature on the plot's x and y axes (position, area, brightness, ...);
- gate with the min/max boxes under the plot (thresholds on the current axes — press Enter or click away to apply) and/or by drawing a lasso around points with the mouse — gated points stay blue, the rest fade out;
- hover a point to see that cell's image crop in the side panel.

`explorer.gated` is the list the next step samples from.

In [ ]:
targets = workflow.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

In [ ]:
explorer = wreact.explore_targets(targets, overviews)
explorer

## 8 · Acquire targets

Type how many targets to acquire and press **Acquire**. That many cells are drawn at random from the gate, re-imaged at the target job, and shown below as pairs — the overview crop (left) and the new high-magnification image (right), both over the same physical window so the cell appears at the same scale in both.

In [ ]:
gallery = wreact.acquire_gallery(
    zmart_controller,
    explorer,
    overviews,
    state=target_state,
    focus=focus,
    after_acquire=lambda records: workflow.hijack_if_simulating(records, simulate=SIMULATE_IMAGES),
)
gallery

## 9 · Summary
Write the summary and run map.

In [ ]:
if not gallery.records:
    raise RuntimeError("no targets have been acquired; use the Acquire button first")

summary = workflow.write_run_report(
    ROOT,
    positions=positions,
    focus=focus,
    overview_records=overview_records,
    targets=gallery.picked,
)
summary

In [ ]:
try:
    if "engine" in globals():
        engine.shutdown()
        del engine
finally:
    zmart_controller.disconnect()
    del zmart_controller